<a href="https://colab.research.google.com/github/Zoye-J/FlyRank--MachineLearning/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zoye-J/FlyRank--MachineLearning/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# Setup - make sure we're in the right directory
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv")
print("Starter data found. You're ready!")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready!


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*


I choose Lane 4 : Search Intent

Why I Chose This:
It addresses a critical gap in content strategy, to understand whether content actually matches what users are searching for. With 54.2% of pages in this dataset showing declining trends, intent mismatch could be a cause of poor performance. Additionally, as someone interested in cybersecurity, I see parallels here with detecting malicious content mismatched intent is often a red flag for SEO poisoning or phishing pages that disguise themselves as legitimate content so understanding intent-content alignment could help both improve content strategy AND detect suspicious patterns.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

My Research Question:
Does content type match search intent, and how does intent-content mismatch affect page performance?


I want to Identify which pages have intent-content mismatch so they can be flagged for review.

Action Someone Can Take can be:
Content Team: Rewrite or re-categorize mismatched content
SEO Team: Adjust targeting to match actual content
Security Team: Flag mismatched pages for potential malicious intent (spoofing, black-hat SEO)

Cost of Getting It Wrong:
False Positive, Flagged as mismatched but is fine, where content team wastes 2-4 hours reviewing/reworking a page unnecessarily.
False Negative, Missed mismatched content, the page continues declining, losing traffic and potential security risk if malicious
Security Context, Malicious intent missed, where users could be exposed to phishing or misleading content.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt


df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Dataset: {df.shape[0]:,} pages, {df.shape[1]} features")

#intent-content distribution
print("INTENT-CONTENT DISTRIBUTION")
intent_by_type = df.groupby(['main_intent', 'content_type']).size().unstack(fill_value=0)
print(f"\n{intent_by_type}")
print(f"\n There are {len(df['main_intent'].unique())} intent types and {len(df['content_type'].unique())} content types")
print(f"We can analyze which combinations are common vs. rare")

#Checking if intent-content mismatch correlates with decline

match_patterns = {
    ('keyword article', 'informational'): 'match',
    ('product page', 'transactional'): 'match',
    ('blog post', 'informational'): 'match',
}

def check_match(row):
    key = (row['content_type'], row['main_intent'])
    return match_patterns.get(key, 'unknown')

df['intent_match'] = df.apply(check_match, axis=1)

# Calculate decline rate by match status
match_decline = df[df['intent_match'] == 'match']['trend_direction'].str.lower().eq('down').mean()
unknown_decline = df[df['intent_match'] == 'unknown']['trend_direction'].str.lower().eq('down').mean()

print(f"\nPages with matched intent-content:  {match_decline*100:.1f}% declining")
print(f"Pages with unknown intent-content:  {unknown_decline*100:.1f}% declining")
print(f"Difference: {(unknown_decline - match_decline)*100:.1f}% more declining among mismatched pages")


# 3. CTR by intent type

visible = df[df['impressions_90d'] >= 100]
ctr_by_intent = visible.groupby('main_intent')['ctr'].mean().sort_values(ascending=False)

print(f"\n{ctr_by_intent.round(3)}")
print(f"\n CTR ranges from {ctr_by_intent.min():.3f} to {ctr_by_intent.max():.3f}")
print(f"   → The best intent type is {ctr_by_intent.idxmax()} ({ctr_by_intent.max():.3f})")
print(f"   → The worst intent type is {ctr_by_intent.idxmin()} ({ctr_by_intent.min():.3f})")
print(f"   → Difference: {ctr_by_intent.max() - ctr_by_intent.min():.3f}")

ctr_by_content = visible.groupby('content_type')['ctr'].mean().sort_values(ascending=False)
print(f"\n{ctr_by_content.round(3)}")
print(f"\n Best content type: {ctr_by_content.idxmax()} ({ctr_by_content.max():.3f})")
print(f"Worst content type: {ctr_by_content.idxmin()} ({ctr_by_content.min():.3f})")

# Security angle: detect potential anomalies

# Find pages with unusual intent-content combinations (potential red flags)
unusual = df.groupby(['main_intent', 'content_type']).size().reset_index(name='count')
unusual['pct'] = unusual['count'] / len(df) * 100
print(f"\nUnusual intent-content combinations (frequency < 0.5%):")
rare_combos = unusual[unusual['pct'] < 0.5].sort_values('pct')
print(rare_combos.to_string(index=False))
print(f"\n {len(rare_combos)} rare combinations found (could indicate malicious content)")


Dataset: 30,000 pages, 44 features
INTENT-CONTENT DISTRIBUTION

content_type   keyword article  comparison article
main_intent                                       
commercial                4612                   0
informational            16538                 697
navigational                46                   0
transactional             5733                   0

 There are 5 intent types and 3 content types
We can analyze which combinations are common vs. rare

Pages with matched intent-content:  57.1% declining
Pages with unknown intent-content:  50.7% declining
Difference: -6.4% more declining among mismatched pages

main_intent
navigational     0.319
transactional    0.276
commercial       0.248
informational    0.241
Name: ctr, dtype: float64

 CTR ranges from 0.241 to 0.319
   → The best intent type is navigational (0.319)
   → The worst intent type is informational (0.241)
   → Difference: 0.078

content_type
feedly article        0.706
keyword article       0.252
compariso

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What I CAN Claim (with Evidence):
Observed: In this dataset, pages with certain intent-content combinations have different decline rates
Measured: CTR varies significantly by intent type and content type
Directional: There appears to be a correlation between intent-content mismatch and declining trends
Decision-support: This analysis can help prioritize which pages to review for content-intent alignment
Security-relevant: Unusual intent-content combinations can be flagged for further investigation

What I CANNOT Claim:
Causal proof: "Mismatched intent causes pages to decline" (correlation ≠ causation)
Google's algorithm: "Google penalizes pages with intent mismatch"
Universal truth: "This pattern holds for all websites" (this is specific to this dataset)
Malicious intent confirmed: "Rare combinations are definitely malicious" (they're just unusual)
Perfect prediction: "I can predict which pages will decline with 100% accuracy"

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.